# Notebook 04 — Model Building and Comparative Evaluation

Four separate algorithm sections matching FYP Chapter 4.5. Each uses **one common full environmental feature set** containing profile curvature, plan curvature and TWI, the same static/temporal inventory, frozen train/holdout manifest, 5 km geographic embargo and training-only preprocessing. Run Sections 0–4 first, then one section per model. The original uploaded notebook is not changed.

## 0. Setup and reproducibility

In [1]:
from pathlib import Path
import json, warnings, hashlib
import numpy as np
import pandas as pd
from IPython.display import display
from pyproj import Transformer
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV
from sklearn.neighbors import BallTree
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, balanced_accuracy_score, confusion_matrix)
from scipy.stats import randint, uniform, loguniform
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

OUTER_FOLDS, INNER_FOLDS = 3, 3
EMBARGO_KM = 5.0
SEED = 2026
BLOCK_KM = 50
RECALL_MIN = 0.70
PRECISION_MIN = 0.35
THRESHOLDS = np.round(np.arange(0.10, 0.901, 0.025), 3)
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'data/processed/modelling/static_model_dataset.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open VS Code in the landslide FYP root directory.')
MODEL_DIR = ROOT / 'data/processed/modelling'
FEATURE_DIR = ROOT / 'data/processed/features'
OUT = ROOT / 'outputs/model_evaluation/final_common_features_v1'
OUT.mkdir(parents=True, exist_ok=True)
print('FYP root:', ROOT)
print('Unified matched comparison outputs:', OUT)
print('Validation: grouped 5 km buffer; NOT strict 50 km actual-block independence')


FYP root: C:\landslide_fyp
Unified matched comparison outputs: C:\landslide_fyp\outputs\model_evaluation\final_common_features_v1
Validation: grouped 5 km buffer; NOT strict 50 km actual-block independence


## 1. Data loading and environmental source validation

In [2]:
SIDECAR = FEATURE_DIR / 'historical_environmental_enhancement_recovery_REVIEW.csv'
ORIGINAL = FEATURE_DIR / 'historical_environmental_enhancement.csv'
if not SIDECAR.is_file():
    raise FileNotFoundError('Reviewed Notebook 02 sidecar required: ' + str(SIDECAR))
if not ORIGINAL.is_file():
    raise FileNotFoundError('Original Notebook 02 sidecar required: ' + str(ORIGINAL))
ENHANCEMENT = {
    'vegetation': ['ndvi_pre_event','ndvi_change_2y'],
    'forest_loss': ['forest_loss_5y_250m_pct'],
    'runoff': ['runoff_7d_mm','runoff_30d_mm'],
    'hydrological_terrain': ['merit_upstream_area_km2','merit_hand_m']
}
E_COLS = [v for group in ENHANCEMENT.values() for v in group]
side = pd.read_csv(SIDECAR, low_memory=False)
original = pd.read_csv(ORIGINAL, low_memory=False)
assert side.sample_id.is_unique and original.sample_id.is_unique
assert side.sample_id.astype(str).equals(original.sample_id.astype(str))
assert set(E_COLS).issubset(side.columns)
for col in original.columns:
    if col not in E_COLS:
        assert original[col].equals(side[col]), f'Provenance changed: {col}'
    else:
        old = pd.to_numeric(original[col], errors='coerce')
        now = pd.to_numeric(side[col], errors='coerce')
        assert np.allclose(old[old.notna()], now[old.notna()], atol=1e-8, rtol=0)
side = side[['sample_id'] + E_COLS].copy()
for col in E_COLS:
    side[col] = pd.to_numeric(side[col], errors='coerce').replace([np.inf,-np.inf],np.nan)
DATASETS = {}
for branch in ('static','temporal'):
    raw = pd.read_csv(MODEL_DIR / f'{branch}_model_dataset.csv', low_memory=False)
    assert raw.sample_id.is_unique
    grp = raw.groupby(['matched_event_id','target']).size().unstack(fill_value=0)
    if not ((grp.get(0,0)==2) & (grp.get(1,0)==1)).all():
        raise AssertionError(f'Invalid 1-case/2-control groups: {branch}')
    if any(c in raw for c in E_COLS):
        raise AssertionError('New sidecar columns already exist in original modelling data.')
    df = raw.merge(side, on='sample_id', validate='one_to_one', how='left',sort=False)
    assert len(df)==len(raw) and df.sample_id.equals(raw.sample_id)
    if df[E_COLS].isna().all(axis=1).any():
        print('Notice: some rows have no observed enhancement; retained for fold-specific imputation.')
    df['target'] = pd.to_numeric(df.target,errors='raise').astype(int)
    for col in ['elevation_m','slope_deg','aspect_sin','aspect_cos','soil_clay_surface',
                'soil_sand_surface','soil_silt_surface','soil_bdod_surface','road_distance_m',
                'waterway_distance_m','regional_fault_distance_m','major_fault_window_distance_m',
                'soil_moisture_7d_m3m3',*([f'rain_{d}d_mm' for d in (1,3,7,14,30)]),*E_COLS]:
        if col in df: df[col] = pd.to_numeric(df[col],errors='coerce').replace([np.inf,-np.inf],np.nan)
    for cat in ('landcover_pre_event_class','lithology_material'):
        if cat in df: df[cat] = df[cat].astype('string').replace({'nan':pd.NA,'None':pd.NA,'':pd.NA}).fillna('Unknown').astype(str)
    DATASETS[branch] = df
    print(branch, 'rows',len(df),'cases',df.target.sum(),'controls',len(df)-df.target.sum())


static rows 1968 cases 656 controls 1312
temporal rows 1929 cases 643 controls 1286


## 2. Restore geographic partition and join terrain features

In [3]:

# ============================================================
# 2. Restore Existing Geographic Holdout Partition
# ============================================================

MANIFEST = (
    ROOT / 'outputs' / 'model_evaluation'
    / 'spatial_partition_manifest.csv'
)

if not MANIFEST.is_file():
    raise FileNotFoundError(
        'Frozen spatial_partition_manifest.csv is required. '
        'Do not generate a new holdout partition.'
    )

# Load the previously established geographic partition.
man = pd.read_csv(
    MANIFEST,
    low_memory=False,
    dtype={
        'sample_id': str,
        'matched_event_id': str,
        'spatial_block_id': str
    }
)

required = {
    'branch',
    'sample_id',
    'matched_event_id',
    'spatial_block_id',
    'partition'
}

if not required.issubset(man.columns):
    raise AssertionError(
        f'Manifest missing columns: {required - set(man.columns)}'
    )

if man.duplicated(['branch', 'sample_id']).any():
    raise AssertionError(
        'Duplicate branch/sample IDs in geographic manifest.'
    )

if not set(man['partition'].dropna().astype(str)) <= {
    'train', 'heldout'
}:
    raise AssertionError(
        'Unexpected partition values in manifest.'
    )

if man['partition'].isna().any():
    raise AssertionError('Missing partition assignments.')

if man['spatial_block_id'].isna().any():
    raise AssertionError('Missing manifest spatial block IDs.')

# Coordinate transformation for geographic validation.
project = Transformer.from_crs(
    'EPSG:4326',
    'EPSG:6933',
    always_xy=True
)

TRAIN = {}

for branch, all_df in DATASETS.items():

    print(f'\nPreparing {branch.upper()} geographic partition')

    # Copy data to preserve original modelling datasets.
    dataset = all_df.copy()
    dataset['sample_id'] = dataset['sample_id'].astype(str)
    dataset['matched_event_id'] = (
        dataset['matched_event_id'].astype(str)
    )

    # The modelling CSV already contains spatial_block_id.
    # Rename it to avoid pandas merge suffixes.
    if 'spatial_block_id' in dataset.columns:
        dataset = dataset.rename(
            columns={
                'spatial_block_id': 'source_spatial_block_id'
            }
        )

    part = man.loc[
        man['branch'].eq(branch)
    ].copy()

    # Check sample identity before merging.
    if set(part['sample_id']) != set(dataset['sample_id']):
        raise AssertionError(
            f'{branch}: Manifest and modelling sample IDs differ.'
        )

    # Keep the frozen manifest spatial block ID as the
    # authoritative grouping column for validation.
    aligned = dataset.merge(
        part[
            [
                'sample_id',
                'matched_event_id',
                'partition',
                'spatial_block_id'
            ]
        ].rename(
            columns={
                'matched_event_id': 'manifest_group'
            }
        ),
        on='sample_id',
        how='left',
        validate='one_to_one',
        sort=False
    )

    # Verify that matched case/control identifiers agree.
    if not (
        aligned['matched_event_id'].astype(str)
        == aligned['manifest_group'].astype(str)
    ).all():
        raise AssertionError(
            f'{branch}: Matched-group provenance differs.'
        )

    # Compare original dataset block IDs with the manifest.
    if 'source_spatial_block_id' in aligned.columns:

        block_mismatch = (
            aligned['source_spatial_block_id'].astype(str)
            != aligned['spatial_block_id'].astype(str)
        )

        print(
            'Dataset/manifest spatial block differences:',
            int(block_mismatch.sum())
        )

        if block_mismatch.any():
            print(
                'Using the frozen manifest block IDs for '
                'geographic validation.'
            )

    # A matched event and its controls must remain together.
    if (
        aligned.groupby('matched_event_id')['partition']
        .nunique()
        .gt(1)
        .any()
    ):
        raise AssertionError(
            f'{branch}: A matched group crosses '
            'the train/heldout boundary.'
        )

    # Extract training rows only.
    # Held-out rows are excluded from all model selection.
    train = (
        aligned.loc[aligned['partition'].eq('train')]
        .drop(
            columns=[
                'partition',
                'manifest_group',
                'source_spatial_block_id'
            ],
            errors='ignore'
        )
        .reset_index(drop=True)
    )

    if train.empty or train['target'].nunique() != 2:
        raise AssertionError(
            f'{branch}: Invalid training partition.'
        )

    if 'spatial_block_id' not in train.columns:
        raise AssertionError(
            f'{branch}: Spatial block ID not preserved.'
        )

    TRAIN[branch] = train

    print('Training samples:', len(train))
    print(
        'Held-out samples:',
        len(aligned) - len(train)
    )
    print(
        'Unique geographic blocks:',
        train['spatial_block_id'].nunique()
    )
    print(
        'Unique matched events:',
        train['matched_event_id'].nunique()
    )

# Confirm the fixed manifest has not been changed.
print(
    '\nManifest SHA256:',
    hashlib.sha256(MANIFEST.read_bytes()).hexdigest()[:20]
)

print('\nGeographic partition restoration completed.')


# 2.1 Add locally derived static terrain values without changing cases/controls.
TERRAIN_CSV = FEATURE_DIR / 'terrain_derivatives_v1.csv'
TERRAIN_CURV = ['profile_curvature_m_inv', 'plan_curvature_m_inv']
TERRAIN_TWI = ['topographic_wetness_index']
TERRAIN_ADDITIONS = {
    'A_baseline': [],
    'B_curvature': TERRAIN_CURV,
    'C_twi': TERRAIN_TWI,
    'D_curvature_twi': TERRAIN_CURV + TERRAIN_TWI
}
if not TERRAIN_CSV.is_file():
    raise FileNotFoundError('Complete Notebook 02 terrain extraction first: '+str(TERRAIN_CSV))
terrain = pd.read_csv(TERRAIN_CSV, dtype={'sample_id':str,'matched_event_id':str})
needed = {'sample_id','matched_event_id','longitude','latitude','curvature_status','twi_status',
          *TERRAIN_CURV,*TERRAIN_TWI}
if not needed.issubset(terrain.columns):
    raise ValueError('Missing terrain sidecar fields: '+str(needed-set(terrain)))
if terrain.sample_id.duplicated().any():
    raise AssertionError('Duplicate terrain sample_id')
for col in TERRAIN_CURV + TERRAIN_TWI:
    terrain[col] = pd.to_numeric(terrain[col], errors='coerce').replace([np.inf,-np.inf], np.nan)
terrain_qa = []
for branch, frame in TRAIN.items():
    raw_frame = frame.reset_index(drop=True).copy()
    raw_frame['sample_id'] = raw_frame.sample_id.astype(str)
    joined = raw_frame.merge(terrain, on='sample_id', how='left', validate='one_to_one',
                            sort=False, suffixes=('','_terrain'))
    if len(joined)!=len(raw_frame) or not joined.sample_id.equals(raw_frame.sample_id):
        raise AssertionError('Terrain merge changed sample count/order')
    if not joined.matched_event_id.astype(str).equals(joined.matched_event_id_terrain.astype(str)):
        raise AssertionError('Terrain merge changed matched event ID')
    for coord in ('longitude','latitude'):
        if not np.allclose(joined[coord],joined[coord+'_terrain'],atol=1e-7,rtol=0):
            raise AssertionError('Terrain coordinate mismatch: '+coord)
    for col in TERRAIN_CURV + TERRAIN_TWI:
        for target, group in joined.groupby('target'):
            measured = float(group[col].notna().mean())
            terrain_qa.append(dict(branch=branch,feature=col,target=int(target),
                                   measured_fraction=measured,count=len(group)))
            if measured < .90:
                raise ValueError(f'{branch} class {target} {col}: {measured:.1%} measured; repair coverage')
    TRAIN[branch] = joined
print('Terrain sidecar joined in memory; original modelling CSVs unchanged.')
display(pd.DataFrame(terrain_qa).round(3))



Preparing STATIC geographic partition
Dataset/manifest spatial block differences: 0
Training samples: 1308
Held-out samples: 660
Unique geographic blocks: 68
Unique matched events: 436

Preparing TEMPORAL geographic partition
Dataset/manifest spatial block differences: 0
Training samples: 1287
Held-out samples: 642
Unique geographic blocks: 67
Unique matched events: 429

Manifest SHA256: aebb5874cc79119ec08b

Geographic partition restoration completed.
Terrain sidecar joined in memory; original modelling CSVs unchanged.


,branch,feature,target,measured_fraction,count
0,static,profile_curvature_m_inv,0,0.946,872
1,static,profile_curvature_m_inv,1,0.993,436
2,static,plan_curvature_m_inv,0,0.946,872
3,static,plan_curvature_m_inv,1,0.993,436
4,static,topographic_wetness_index,0,0.999,872
5,static,topographic_wetness_index,1,1.000,436
6,temporal,profile_curvature_m_inv,0,0.945,858
7,temporal,profile_curvature_m_inv,1,0.993,429
8,temporal,plan_curvature_m_inv,0,0.945,858
9,temporal,plan_curvature_m_inv,1,0.993,429


## 3. Final shared environmental feature selection
The final baseline here means **all selected environmental predictors including curvature and TWI**, not the pre-terrain reduced baseline. Static and temporal branches have different time-dependent feature lists, but all algorithms within each branch use identical columns.

In [4]:
# ONE predefined, identical feature set for every algorithm within each branch.
# All available, previously validated environmental groups are included, except
# lithology/fault predictors excluded in the established protocol.
# Weather and runoff are included only for the temporal branch.
COMMON_FEATURES = {
    'static': [
        'elevation_m','slope_deg','aspect_sin','aspect_cos',
        'profile_curvature_m_inv','plan_curvature_m_inv','topographic_wetness_index',
        'soil_clay_surface','soil_sand_surface','soil_silt_surface','soil_bdod_surface',
        'road_distance_m','waterway_distance_m','landcover_pre_event_class',
        'ndvi_pre_event','ndvi_change_2y','forest_loss_5y_250m_pct',
        'merit_upstream_area_km2','merit_hand_m'
    ],
    'temporal': [
        'elevation_m','slope_deg','aspect_sin','aspect_cos',
        'profile_curvature_m_inv','plan_curvature_m_inv','topographic_wetness_index',
        'soil_clay_surface','soil_sand_surface','soil_silt_surface','soil_bdod_surface',
        'road_distance_m','waterway_distance_m','landcover_pre_event_class',
        'ndvi_pre_event','ndvi_change_2y','forest_loss_5y_250m_pct',
        'merit_upstream_area_km2','merit_hand_m',
        'rain_1d_mm','rain_3d_mm','rain_7d_mm','rain_14d_mm','rain_30d_mm',
        'soil_moisture_7d_m3m3','runoff_7d_mm','runoff_30d_mm'
    ]
}
assert all(len(cols)==len(set(cols)) for cols in COMMON_FEATURES.values())
for branch, df in TRAIN.items():
    cols=COMMON_FEATURES[branch]
    missing=[c for c in cols if c not in df.columns]
    if missing: raise ValueError(f'{branch}: missing environmental columns {missing}')
    print(f'{branch}: {len(df)} training records, {len(cols)} common predictors')
    display(pd.DataFrame({'feature':cols,'measured_pct':[round(100*df[x].notna().mean(),2) for x in cols]}))
print('Each model in a given branch receives exactly the same feature columns and samples.')


static: 1308 training records, 19 common predictors


,feature,measured_pct
0,elevation_m,100.00
1,slope_deg,100.00
2,aspect_sin,100.00
3,aspect_cos,100.00
4,profile_curvature_m_inv,96.18
5,plan_curvature_m_inv,96.18
6,topographic_wetness_index,99.92
7,soil_clay_surface,100.00
8,soil_sand_surface,100.00
9,soil_silt_surface,100.00


temporal: 1287 training records, 27 common predictors


,feature,measured_pct
0,elevation_m,100.00
1,slope_deg,100.00
2,aspect_sin,100.00
3,aspect_cos,100.00
4,profile_curvature_m_inv,96.11
5,plan_curvature_m_inv,96.11
6,topographic_wetness_index,99.92
7,soil_clay_surface,100.00
8,soil_sand_surface,100.00
9,soil_silt_surface,100.00


Each model in a given branch receives exactly the same feature columns and samples.


## 4. Frozen geographic cross-validation and model helper functions
5 km actual-coordinate embargo with matched-triplet purging; 50 km parent-block grouping is not strict actual-block disjointness.

In [5]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import BallTree
NESTED_SEED = SEED
MIN_TRAIN_GROUPS = 10
STRICT_ACTUAL_BLOCK_DISJOINT = False  # EXPLICIT study protocol: grouped 5 km buffer; 50 km actual-block overlap audited, not forbidden
EARTH_KM = 6371.0088
def actual_block_ids(frame):
    xx,yy=project.transform(frame.longitude.to_numpy(float),frame.latitude.to_numpy(float))
    return (np.floor(np.asarray(xx)/(BLOCK_KM*1000)).astype(int).astype(str)+'_'+
            np.floor(np.asarray(yy)/(BLOCK_KM*1000)).astype(int).astype(str))

def geo_points(frame):
    if frame[['latitude','longitude']].isna().any().any():
        raise ValueError('Missing coordinates would invalidate geographic separation')
    return np.radians(frame[['latitude','longitude']].to_numpy(dtype=float))

def make_embargo_folds(frame,n_splits,seed,stage,branch,embargo_km=None,strict_actual=None):
    """Return group-preserving folds, geographic-distance checks, and purge audit.

    Validation uses the frozen parent-case block assignments. Complete matched
    groups in candidate training are purged if ANY member is within the 5 km
    embargo or shares an actual-coordinate 50 km block with validation.
    Strict 50 km actual-block disjointness is NOT used in this training run;
    any such overlaps are explicitly audited. No automatic protocol switching.
    """
    embargo_km = EMBARGO_KM if embargo_km is None else float(embargo_km)
    strict_actual = STRICT_ACTUAL_BLOCK_DISJOINT if strict_actual is None else bool(strict_actual)
    frame=frame.reset_index(drop=True)
    group=frame['spatial_block_id'].astype(str)
    raw=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=seed)
    coords=geo_points(frame)
    actual=np.asarray(actual_block_ids(frame))
    # Require the parent-case grouping itself to preserve every matched group.
    if frame.assign(_parent_block=group).groupby('matched_event_id')['_parent_block'].nunique().gt(1).any():
        raise AssertionError('A matched group has multiple parent spatial block IDs')
    folds=[]; audits=[]
    for fold_id,(tr,va) in enumerate(raw.split(frame,frame.target,groups=group),1):
        valid_actual=set(actual[va]); overlap=np.isin(actual[tr],list(valid_actual))
        km=BallTree(coords[va],metric='haversine').query(coords[tr],k=1)[0][:,0]*EARTH_KM
        bad=(km<embargo_km-1e-8) | (overlap if strict_actual else False)
        purged_groups=set(frame.iloc[tr[bad]].matched_event_id.astype(str))
        keep=~frame.iloc[tr].matched_event_id.astype(str).isin(purged_groups).to_numpy()
        tr_p=tr[keep]
        if len(tr_p)==0 or len(va)==0 or frame.iloc[tr_p].target.nunique()!=2 or frame.iloc[va].target.nunique()!=2:
            raise ValueError(f'{branch}/{stage}/fold {fold_id}: embargo left a single class or no training data')
        if frame.iloc[tr_p].matched_event_id.nunique()<MIN_TRAIN_GROUPS:
            raise ValueError(f'{branch}/{stage}/fold {fold_id}: too few matched groups')
        if set(frame.iloc[tr_p].matched_event_id) & set(frame.iloc[va].matched_event_id):
            raise AssertionError('Matched case/control group crossed fold boundary')
        shared_actual_blocks=len(set(actual[tr_p]) & set(actual[va]))
        if frame.iloc[tr_p].groupby('matched_event_id').size().ne(3).any():
            raise AssertionError('Purging split a training matched group')
        if frame.iloc[va].groupby('matched_event_id').size().ne(3).any():
            raise AssertionError('Validation split a matched group')
        if strict_actual and shared_actual_blocks:
            raise AssertionError('Actual-location spatial blocks overlap under strict mode')
        nearest=float(BallTree(coords[va],metric='haversine').query(coords[tr_p],k=1)[0].min()*EARTH_KM)
        if nearest+1e-7<embargo_km:
            raise AssertionError('Spatial embargo failed')
        folds.append((tr_p,va))
        audits.append(dict(branch=branch,stage=stage,seed=seed,fold=fold_id,
            validation_protocol=('strict_50km_plus_5km_buffer' if strict_actual else 'grouped_5km_buffer'),
            candidate_train_rows=len(tr),train_rows=len(tr_p),validation_rows=len(va),
            removed_rows=len(tr)-len(tr_p),removed_matched_groups=len(purged_groups),
            removal_fraction=float((len(tr)-len(tr_p))/len(tr)),
            train_groups=frame.iloc[tr_p].matched_event_id.nunique(),
            validation_groups=frame.iloc[va].matched_event_id.nunique(),
            train_positive=int(frame.iloc[tr_p].target.sum()),
            train_background=int((frame.iloc[tr_p].target==0).sum()),
            validation_positive=int(frame.iloc[va].target.sum()),
            validation_background=int((frame.iloc[va].target==0).sum()),
            min_actual_train_valid_km=nearest,shared_actual_location_blocks=shared_actual_blocks,
            embargo_km=embargo_km,strict_actual_block_disjoint=strict_actual,
            shared_matched_groups=len(set(frame.iloc[tr_p].matched_event_id) & set(frame.iloc[va].matched_event_id)),
            complete_training_triplets=bool(frame.iloc[tr_p].groupby('matched_event_id').size().eq(3).all()),
            complete_validation_triplets=bool(frame.iloc[va].groupby('matched_event_id').size().eq(3).all())))
    return folds,pd.DataFrame(audits)


In [6]:
from sklearn.model_selection import GridSearchCV
# One algorithm per section. Shared functions guarantee identical feature inputs,
# spatial folds, metrics, model-selection policy and outputs.
RUN_FINAL_MODEL_TRAINING = True
RUN_ONE_FOLD_TEST = False  # Set True for a quick smoke test (does not write final files).
FINAL_OUT = ROOT / 'outputs/model_evaluation/final_common_features_v1'
MODEL_NAMES = ('LogisticRegression','LinearSVM','RandomForest','XGBoost')
METRICS = ['roc_auc','pr_auc','accuracy','precision','recall','f1',
           'balanced_accuracy','train_validation_auc_gap']
if (OUTER_FOLDS,INNER_FOLDS,SEED,EMBARGO_KM,BLOCK_KM)!=(3,3,2026,5.,50):
    raise AssertionError('Frozen geographical validation settings changed.')
if STRICT_ACTUAL_BLOCK_DISJOINT is not False:
    raise AssertionError('This study uses buffered CV, not strict block-disjoint CV.')
if XGBClassifier is None:
    print('XGBoost is not installed; the XGBoost section will require installation.')

def build_model(kind,columns):
    numerical=[x for x in columns if x!='landcover_pre_event_class']
    categorical=[x for x in columns if x=='landcover_pre_event_class']
    steps=[('imputer',SimpleImputer(strategy='median',keep_empty_features=True))]
    if kind in ('LogisticRegression','LinearSVM'):
        steps.append(('scaler',StandardScaler()))
    ct=[('numeric',Pipeline(steps),numerical)]
    if categorical:
        ct.append(('categorical',Pipeline([
            ('imputer',SimpleImputer(strategy='constant',fill_value='Unknown')),
            ('onehot',OneHotEncoder(handle_unknown='ignore'))]),categorical))
    pre=ColumnTransformer(ct,remainder='drop',sparse_threshold=0)
    if kind=='LogisticRegression':
        est=LogisticRegression(max_iter=3500,class_weight='balanced',random_state=SEED)
        params={'model__C':[0.03,0.1,1.0]}
    elif kind=='LinearSVM':
        est=SVC(kernel='linear',class_weight='balanced',random_state=SEED)
        params={'model__C':[0.03,0.1,1.0]}
    elif kind=='RandomForest':
        est=RandomForestClassifier(n_estimators=250,random_state=SEED,n_jobs=2,
                                   class_weight='balanced_subsample')
        params={'model__max_depth':[3,5,8], 'model__min_samples_leaf':[10,20]}
    elif kind=='XGBoost':
        if XGBClassifier is None:raise ImportError('Install xgboost in your notebook environment.')
        est=XGBClassifier(n_estimators=200,random_state=SEED,n_jobs=2,
                          eval_metric='logloss',tree_method='hist',scale_pos_weight=2)
        params={'model__max_depth':[2,3],'model__learning_rate':[0.03,0.07],
                'model__min_child_weight':[5,15]}
    else:raise ValueError(kind)
    return Pipeline([('prep',pre),('model',est)]),params

def model_scores(fitted,frame,columns,kind):
    if kind=='LinearSVM':
        score=np.asarray(fitted.decision_function(frame[columns]),dtype=float)
        return score,(score>=0).astype(int),0.0
    score=np.asarray(fitted.predict_proba(frame[columns])[:,1],dtype=float)
    return score,(score>=0.5).astype(int),0.5

# Prepare / audit the exact same folds once, before any model is fitted.
FOLD_TASKS={}
GEO_AUDITS=[]
for branch,frame in TRAIN.items():
    outer,audit=make_embargo_folds(frame,OUTER_FOLDS,SEED,'outer_final',branch)
    GEO_AUDITS.append(audit)
    for fold,(tr_idx,va_idx) in enumerate(outer,1):
        if RUN_ONE_FOLD_TEST and fold!=1:continue
        tr=frame.iloc[tr_idx].reset_index(drop=True)
        va=frame.iloc[va_idx].reset_index(drop=True)
        inner,iaudit=make_embargo_folds(tr,INNER_FOLDS,SEED+fold,'inner_final',branch)
        GEO_AUDITS.append(iaudit)
        FOLD_TASKS[(branch,fold)]=(tr,va,inner,
            hashlib.sha256('|'.join(va.sample_id.astype(str)).encode()).hexdigest()[:16])
if not RUN_ONE_FOLD_TEST:
    FINAL_OUT.mkdir(parents=True,exist_ok=True)
    audit_df=pd.concat(GEO_AUDITS,ignore_index=True)
    audit_df.to_csv(FINAL_OUT/'01_geographic_audit.csv',index=False)
print('Prepared folds:',list(FOLD_TASKS))

def run_one_algorithm(kind):
    if not RUN_FINAL_MODEL_TRAINING:
        print('Training disabled for',kind);return None
    target=FINAL_OUT/f'{kind}_folds.csv'
    if not RUN_ONE_FOLD_TEST and target.exists():
        existing=pd.read_csv(target)
        required={(b,f) for b in TRAIN for f in range(1,OUTER_FOLDS+1)}
        observed=set(zip(existing.branch,existing.outer_fold.astype(int)))
        if observed != required or existing.duplicated(['branch','outer_fold']).any():
            raise ValueError(f'{target} has incomplete/duplicated folds; archive it before rerun')
        if not (existing.feature_signature == existing.apply(
            lambda r: hashlib.sha256('|'.join(COMMON_FEATURES[r.branch]).encode()).hexdigest()[:16], axis=1)).all():
            raise ValueError(f'{target} has a different feature configuration')
        print('Reusing existing complete model result:',target)
        rows=existing
    else:
        observations=[]
        for (branch,fold),(tr,va,inner,signature) in FOLD_TASKS.items():
            cols=COMMON_FEATURES[branch]
            pipe,params=build_model(kind,cols)
            tune=GridSearchCV(pipe,params,cv=inner,scoring='roc_auc',refit=True,
                              error_score='raise',n_jobs=1)
            tune.fit(tr[cols],tr.target)
            fit=tune.best_estimator_
            score,pred,threshold=model_scores(fit,va,cols,kind)
            train_score,_,_=model_scores(fit,tr,cols,kind)
            y=va.target.to_numpy()
            tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
            test_auc=roc_auc_score(y,score)
            observations.append(dict(branch=branch,algorithm=kind,outer_fold=fold,
                validation_sample_hash=signature,
                feature_signature=hashlib.sha256('|'.join(cols).encode()).hexdigest()[:16],
                predictors=len(cols),train_rows=len(tr),validation_rows=len(va),
                best_params=json.dumps(tune.best_params_,sort_keys=True),
                inner_roc_auc=tune.best_score_,threshold=threshold,
                train_roc_auc=roc_auc_score(tr.target,train_score),
                roc_auc=test_auc,pr_auc=average_precision_score(y,score),
                accuracy=accuracy_score(y,pred),
                precision=precision_score(y,pred,zero_division=0),
                recall=recall_score(y,pred,zero_division=0),
                f1=f1_score(y,pred,zero_division=0),
                balanced_accuracy=balanced_accuracy_score(y,pred),
                train_validation_auc_gap=roc_auc_score(tr.target,train_score)-test_auc,
                tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp)))
            print(kind,branch,'Fold',fold,'ROC-AUC:',round(test_auc,4),flush=True)
        rows=pd.DataFrame(observations)
        if not RUN_ONE_FOLD_TEST:rows.to_csv(target,index=False)
    by_branch=rows.groupby(['branch','algorithm']).agg(
        **{k+'_mean':(k,'mean') for k in METRICS},
        **{k+'_sd':(k,'std') for k in METRICS},
        folds=('outer_fold','nunique')).reset_index()
    if not RUN_ONE_FOLD_TEST:
        by_branch.to_csv(FINAL_OUT/f'{kind}_summary.csv',index=False)
    print('\n'+kind+' — fold-level validation:')
    display(rows[['branch','outer_fold','predictors','roc_auc','pr_auc','accuracy',
                  'precision','recall','f1','balanced_accuracy',
                  'train_validation_auc_gap']].round(4))
    print(kind+' — branch summary (mean ± SD across folds):')
    display(by_branch[['branch','roc_auc_mean','roc_auc_sd','pr_auc_mean',
                       'accuracy_mean','precision_mean','recall_mean',
                       'f1_mean','balanced_accuracy_mean',
                       'train_validation_auc_gap_mean']].round(4))
    return by_branch


Prepared folds: [('static', 1), ('static', 2), ('static', 3), ('temporal', 1), ('temporal', 2), ('temporal', 3)]


## 5.1 Logistic Regression — model training and results
Report the static and temporal metrics in Chapter 4.5.1; interpret independently under Chapter 5.2.

In [7]:
RESULT_LOGISTICREGRESSION = run_one_algorithm('LogisticRegression')


LogisticRegression static Fold 1 ROC-AUC: 0.7964
LogisticRegression static Fold 2 ROC-AUC: 0.8284
LogisticRegression static Fold 3 ROC-AUC: 0.7049
LogisticRegression temporal Fold 1 ROC-AUC: 0.7813
LogisticRegression temporal Fold 2 ROC-AUC: 0.853
LogisticRegression temporal Fold 3 ROC-AUC: 0.698

LogisticRegression — fold-level validation:


,branch,outer_fold,predictors,roc_auc,pr_auc,accuracy,precision,recall,f1,balanced_accuracy,train_validation_auc_gap
0,static,1,19,0.7964,0.6201,0.6872,0.5187,0.8562,0.6460,0.7295,0.0956
1,static,2,19,0.8284,0.7171,0.7425,0.5829,0.8000,0.6744,0.7569,-0.0003
2,static,3,19,0.7049,0.5180,0.6667,0.5000,0.5310,0.5151,0.6328,0.1956
3,temporal,1,27,0.7813,0.6141,0.6503,0.4859,0.8462,0.6173,0.6993,0.1279
4,temporal,2,27,0.8530,0.7220,0.8205,0.7012,0.8042,0.7492,0.8164,-0.0518
5,temporal,3,27,0.6980,0.4986,0.6294,0.4588,0.6224,0.5282,0.6276,0.2176


LogisticRegression — branch summary (mean ± SD across folds):


,branch,roc_auc_mean,roc_auc_sd,pr_auc_mean,accuracy_mean,precision_mean,recall_mean,f1_mean,balanced_accuracy_mean,train_validation_auc_gap_mean
0,static,0.7766,0.0641,0.6184,0.6988,0.5339,0.7291,0.6118,0.7064,0.0969
1,temporal,0.7775,0.0776,0.6116,0.7001,0.5486,0.7576,0.6316,0.7145,0.0979


## 5.2 Random Forest — model training and results
Same predictors, folds, and target observations as Logistic Regression.

In [8]:
RESULT_RANDOMFOREST = run_one_algorithm('RandomForest')


RandomForest static Fold 1 ROC-AUC: 0.7952
RandomForest static Fold 2 ROC-AUC: 0.6036
RandomForest static Fold 3 ROC-AUC: 0.6274
RandomForest temporal Fold 1 ROC-AUC: 0.7767
RandomForest temporal Fold 2 ROC-AUC: 0.6057
RandomForest temporal Fold 3 ROC-AUC: 0.675

RandomForest — fold-level validation:


,branch,outer_fold,predictors,roc_auc,pr_auc,accuracy,precision,recall,f1,balanced_accuracy,train_validation_auc_gap
0,static,1,19,0.7952,0.5864,0.7260,0.5644,0.7808,0.6552,0.7397,0.1805
1,static,2,19,0.6036,0.5478,0.6897,0.5463,0.4069,0.4664,0.6190,0.3602
2,static,3,19,0.6274,0.4046,0.6552,0.4731,0.3034,0.3697,0.5672,0.3381
3,temporal,1,27,0.7767,0.5873,0.7203,0.5602,0.7483,0.6407,0.7273,0.2152
4,temporal,2,27,0.6057,0.4638,0.6807,0.5263,0.4196,0.4669,0.6154,0.3692
5,temporal,3,27,0.6750,0.4638,0.6480,0.4615,0.3357,0.3887,0.5699,0.3085


RandomForest — branch summary (mean ± SD across folds):


,branch,roc_auc_mean,roc_auc_sd,pr_auc_mean,accuracy_mean,precision_mean,recall_mean,f1_mean,balanced_accuracy_mean,train_validation_auc_gap_mean
0,static,0.6754,0.1044,0.513,0.6903,0.5279,0.4971,0.4971,0.6420,0.2929
1,temporal,0.6858,0.0860,0.505,0.6830,0.5160,0.5012,0.4988,0.6375,0.2976


## 5.3 XGBoost — model training and results
Same predictors, folds, and target observations as Logistic Regression.

In [9]:
RESULT_XGBOOST = run_one_algorithm('XGBoost')


XGBoost static Fold 1 ROC-AUC: 0.7897
XGBoost static Fold 2 ROC-AUC: 0.6676
XGBoost static Fold 3 ROC-AUC: 0.6838
XGBoost temporal Fold 1 ROC-AUC: 0.7616
XGBoost temporal Fold 2 ROC-AUC: 0.657
XGBoost temporal Fold 3 ROC-AUC: 0.7074

XGBoost — fold-level validation:


,branch,outer_fold,predictors,roc_auc,pr_auc,accuracy,precision,recall,f1,balanced_accuracy,train_validation_auc_gap
0,static,1,19,0.7897,0.5812,0.7466,0.5926,0.7671,0.6687,0.7517,0.2077
1,static,2,19,0.6676,0.5823,0.6345,0.4539,0.4759,0.4646,0.5948,0.2811
2,static,3,19,0.6838,0.4604,0.6575,0.4841,0.4207,0.4502,0.5983,0.3154
3,temporal,1,27,0.7616,0.5613,0.6760,0.5085,0.8392,0.6332,0.7168,0.2322
4,temporal,2,27,0.6570,0.5664,0.5991,0.4181,0.5175,0.4625,0.5787,0.2768
5,temporal,3,27,0.7074,0.5222,0.6760,0.5128,0.5594,0.5351,0.6469,0.2743


XGBoost — branch summary (mean ± SD across folds):


,branch,roc_auc_mean,roc_auc_sd,pr_auc_mean,accuracy_mean,precision_mean,recall_mean,f1_mean,balanced_accuracy_mean,train_validation_auc_gap_mean
0,static,0.7137,0.0663,0.5413,0.6795,0.5102,0.5546,0.5278,0.6483,0.2681
1,temporal,0.7087,0.0523,0.5500,0.6503,0.4798,0.6387,0.5436,0.6474,0.2611


## 5.4 Linear SVM — model training and results
The Linear SVM decision boundary is zero (other models use probability threshold 0.5); ranking metrics are more threshold-independent.

In [10]:
RESULT_LINEARSVM = run_one_algorithm('LinearSVM')


LinearSVM static Fold 1 ROC-AUC: 0.7998
LinearSVM static Fold 2 ROC-AUC: 0.8408
LinearSVM static Fold 3 ROC-AUC: 0.7295
LinearSVM temporal Fold 1 ROC-AUC: 0.7845
LinearSVM temporal Fold 2 ROC-AUC: 0.8576
LinearSVM temporal Fold 3 ROC-AUC: 0.7045

LinearSVM — fold-level validation:


,branch,outer_fold,predictors,roc_auc,pr_auc,accuracy,precision,recall,f1,balanced_accuracy,train_validation_auc_gap
0,static,1,19,0.7998,0.6310,0.6781,0.5103,0.8493,0.6375,0.7209,0.0938
1,static,2,19,0.8408,0.7411,0.7425,0.5813,0.8138,0.6782,0.7603,-0.0102
2,static,3,19,0.7295,0.5555,0.6828,0.5223,0.5655,0.5430,0.6534,0.1714
3,temporal,1,27,0.7845,0.6116,0.6573,0.4919,0.8531,0.6240,0.7063,0.1229
4,temporal,2,27,0.8576,0.7300,0.8089,0.6871,0.7832,0.7320,0.8024,-0.0529
5,temporal,3,27,0.7045,0.5019,0.6597,0.4924,0.6783,0.5706,0.6643,0.2103


LinearSVM — branch summary (mean ± SD across folds):


,branch,roc_auc_mean,roc_auc_sd,pr_auc_mean,accuracy_mean,precision_mean,recall_mean,f1_mean,balanced_accuracy_mean,train_validation_auc_gap_mean
0,static,0.7901,0.0563,0.6425,0.7011,0.5380,0.7429,0.6196,0.7116,0.0850
1,temporal,0.7822,0.0766,0.6145,0.7086,0.5571,0.7716,0.6422,0.7244,0.0934


## 6. Final model comparison for Chapter 5.2
Display only final shared-feature results here. Earlier pre-terrain metrics should be discussed as historical exploratory benchmarks, **not** as a controlled terrain-only improvement unless their other settings are aligned.

In [11]:
# Final fair algorithm comparison: SAME features, SAME observations, SAME folds.
if RUN_ONE_FOLD_TEST:
    print('Smoke test only; full summary is not available.')
else:
    available=[]
    for kind in MODEL_NAMES:
        file=FINAL_OUT/f'{kind}_summary.csv'
        if file.exists():available.append(pd.read_csv(file))
    if len(available)!=len(MODEL_NAMES):
        print('Model sections still pending:',len(MODEL_NAMES)-len(available))
    else:
        table=pd.concat(available,ignore_index=True)
        display(table[['branch','algorithm','roc_auc_mean','roc_auc_sd','pr_auc_mean',
                       'recall_mean','f1_mean','balanced_accuracy_mean',
                       'train_validation_auc_gap_mean']].round(4))
        table.to_csv(FINAL_OUT/'02_final_algorithm_comparison.csv',index=False)
        print('Saved final table:',FINAL_OUT/'02_final_algorithm_comparison.csv')
print('Previous benchmarks are preserved separately; do not treat them as a controlled before/after terrain comparison unless all other settings match.')


,branch,algorithm,roc_auc_mean,roc_auc_sd,pr_auc_mean,recall_mean,f1_mean,balanced_accuracy_mean,train_validation_auc_gap_mean
0,static,LogisticRegression,0.7766,0.0641,0.6184,0.7291,0.6118,0.7064,0.0969
1,temporal,LogisticRegression,0.7775,0.0776,0.6116,0.7576,0.6316,0.7145,0.0979
2,static,LinearSVM,0.7901,0.0563,0.6425,0.7429,0.6196,0.7116,0.0850
3,temporal,LinearSVM,0.7822,0.0766,0.6145,0.7716,0.6422,0.7244,0.0934
4,static,RandomForest,0.6754,0.1044,0.5130,0.4971,0.4971,0.6420,0.2929
5,temporal,RandomForest,0.6858,0.0860,0.5050,0.5012,0.4988,0.6375,0.2976
6,static,XGBoost,0.7137,0.0663,0.5413,0.5546,0.5278,0.6483,0.2681
7,temporal,XGBoost,0.7087,0.0523,0.5500,0.6387,0.5436,0.6474,0.2611


Saved final table: C:\landslide_fyp\outputs\model_evaluation\final_common_features_v1\02_final_algorithm_comparison.csv
Previous benchmarks are preserved separately; do not treat them as a controlled before/after terrain comparison unless all other settings match.
